# သင်ခန်းစာ 09 - ကိုယ်တိုင်၏ အတွေးအခေါ်ကို သတိပြု၍ သုံးသပ်ခြင်း ဒီဇိုင်းပုံစံ


## Setup

ဒီ notebook သည် Microsoft Agent Framework ကို အသုံးပြု၍ Metacognition ဒီဇိုင်းပုံစံကို ဖော်ပြသည်။

**လိုအပ်ချက်များ:**
- Azure OpenAI deployment ကို environment variables များဖြင့် ပြင်ဆင်ထားရမည်
- Azure CLI ကို အတည်ပြုထားရမည် (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Metacognition ဆိုတာဘာလဲ?

Metacognition သည် **အတွေးအကြောင်း တွေးခြင်း** ဖြစ်သည်။ In the context of AI agents, it means building agents that can:

- **ကိုယ်တိုင် သုံးသပ်ခြင်း** သူတို့၏ ထွက်လာသော ဖြေရှင်းချက်များနှင့် ရှင်းလင်းချက် လုပ်ငန်းစဉ်များအား သုံးသပ်နိုင်ခြင်း
- **အမှားများကို တွေ့ရှိခြင်း** နှင့် တိတ်တိတ်ပျက်ကွက်သွားခြင်း မဖြစ်စေဘဲ သက်သာစွာ ပြန်လည်ထိန်းသိမ်းနိုင်ခြင်း
- **အကဲဖြတ်ခြင်း** သူတို့၏ ဖြေရှင်းချက်များ ပြည့်စုံကောင်းမွန်မှုနှင့် အထောက်အကူရှိမှန်း စစ်ဆေးနိုင်ခြင်း
- **နည်းဗျူဟာပြောင်းလဲနိုင်ခြင်း** မူလနည်းလမ်း မအောင်မြင်ပါက မဟာဗျူဟာကို ပြောင်းလဲ သို့မဟုတ် backup system ကဲ့သို့သော အစားထိုး စနစ်သို့ ပြန်ရွေ့နိုင်ခြင်း (ဥပမာ)

A metacognitive agent doesn't just answer questions — it monitors its own performance and adjusts on the fly.


## အဓိကနှင့် အစားထိုး ကိရိယာများ

ကိုယ်တွေးခေါ်ခြင်း (metacognition) ပုံစံတစ်ခုအဖြစ် အများအပြား တွေ့ရတာက **အစားထိုး မဟာဗျူဟာ** ဖြစ်သည်။ အေးဂျင့်သည် အရင်ဆုံး အဓိက ကိရိယာကို စမ်းကြည့်ပြီး၊ ၎င်းသည် မအောင်မြင်ခဲ့ပါက (ဥပမာ - 404 အမှား) အဲဒီအမှားကို သဘောပေါက်ကာ ဖွင့်မြင်စွာ အစားထိုး ကိရိယာသို့ ပြောင်းရွှေ့သည်။

ဤသည်မှာ အဓိက ဝန်ဆောင်မှုများ မရရှိနိုင်နိုင်သော အခြေအနေပေါ်တွင် တွေ့ရသော အမှန်တကယ် စနစ်များနှင့် ကိုက်ညီသည်။ အေးဂျင့်များသည် ပြဿနာကို ကိုယ်တိုင် ရှာဖွေ၍ အခြားရွေ့လျားနိုင်သည့် လမ်းကြောင်းတစ်ခုကို ရွေးချယ်ရမည် ဖြစ်သည်။

Below we define two flight lookup tools:
- **အဓိက** — ပဲရီစ်၊ တိုကျိုနှင့် ဘာဆီလိုနာကို ဖုံးလွှမ်းသည်
- **အစားထိုး** — ဘာလင်၊ ဆင်ဒနီနှင့် နယူးယောက်မြို့ကို ဖုံးလွှမ်းသည်


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## ကိုယ်ကို ပြန်လေ့လာနိုင်ပြီး အမှားပြန်လည်ကယ်ဆယ်နိုင်သော အေးဂျင့်

အောက်ပါ အေးဂျင့်ကို အဓိက ပျံသန်းစနစ်ကို ပထမဦး စွာ စမ်းသပ်ရန်၊ မအောင်မြင်မှုများကို သိမြင်ရန်၊ နှင့် ဖော်ပြသာယာ စွာ အစားထိုး စနစ်သို့ ပြန်လှည့်ရန် ညွှန်ကြားထားသည်။ တစ်ခုချင်းစီ၏ တုံ့ပြန်ချက်ပြီးနောက်၌ ၎င်းသည် အသုံးပြုသူ၏ မေးခွန်းကို အပြည့်အဝ ဖြေရှင်းပေးနိုင်ခဲ့သလား ဆိုတာကို အတိုချုံး ကိုယ်အကဲဖြတ် ပြုလုပ်သည်။


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## ကိုယ်တိုင်အကဲဖြတ်မှု ပုံစံ

မိမိစဉ်းစားမှု (metacognition) ၏ နောက်ထပ်ဖက်တစ်ခုမှာ **ကိုယ်တိုင်အကဲဖြတ်ခြင်း** ဖြစ်သည်။ အခြား အေးဂျင့်တစ်ဦး (သို့မဟုတ် တစ်ကြိမ်ထပ်၍ တူညီသော အေးဂျင့်) သည် တုံ့ပြန်ချက်ကို ပြည့်စုံမှု၊ တိကျမှန်ကန်မှု နှင့် အသုံးဝင်မှု အရ ပြန်လည်ဆန်းစစ်သည်။

အောက်တွင် ကျွန်ုပ်တို့သည် ခရီးသွား အေးဂျင့်၏ တုံ့ပြန်ချက်များကို သုံးအတိုင်းအတာပေါ်အရ အမှတ်ပေးသည့် `ResponseEvaluator` အေးဂျင့်ကို ဖန်တီးသည်။


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## အကျဉ်းချုံး

ဤသင်ခန်းစာတွင် သင်သည် Microsoft Agent Framework ကို အသုံးပြု၍ **ကိုယ်တိုင်စဉ်းစားနိုင်သော အေးဂျင့်များ** ကို မည်သို့ တည်ဆောက်ရသည်ကို သင်ယူခဲ့ပါသည်။

- **ကိုယ်အတွေးသုံးသပ်ခြင်း**: အေးဂျင့်များသည် ၎င်းတို့၏ ကိုယ်ပိုင် အတွေးအခေါ်များကို စောင့်ကြည့်ပြီး ဖြစ်ပျက်ခဲ့သောအချက်များကို ပွင့်လင်းမြင်သာစွာ ဆက်သွယ်ပြောကြားသည်။
- **အမှားပြန်လည်ကောင်းမွန်ရေးနှင့် အစားထိုးနည်းများ**: အဓိက + backup ကိရိယာ ပုံစံဖြင့် အေးဂျင့်သည် အကျဉ်းချုပ်မှုများကို တွေ့ရှိပြီး (ဥပမာ၊ 404 အမှား) အလိုအလျောက် အခြားအရင်းအမြစ်တစ်ခုကို စမ်းသပ်ကြည့်သည်။
- **ကိုယ်တိုင်အကဲဖြတ်ခြင်း**: တစ်ဦး သီးခြားအကဲဖြတ်သူ အေးဂျင့်တစ်ခုက တုံ့ပြန်ချက်များကို ပြည့်စုံမှု၊ မှန်ကန်မှုနှင့် အသုံးဝင်မှုအရ အမှတ်ပေးသည်။

ဤပုံစံများကြောင့် အေးဂျင့်များသည် ပိုမိုခိုင်မာ၊ ပိုမိုပွင့်လင်းမြင်သာ၊ နှင့် ယုံကြည်စိတ်ချရ မှတ်ချက်များရှိလာပြီး — ထုတ်ကုန်အဆင့်တွင် အသုံးချရန် အလွန်အရေးပါတဲ့ အရည်အချင်းများဖြစ်သည်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
သတိပေးချက်:
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) ဖြင့် ဘာသာပြန်ထားပါသည်။ ကျွန်တော်/ကျွန်မတို့သည် တိကျမှုအတွက် ကြိုးစားပေမယ့် အလိုအလျောက်ဘာသာပြန်ချက်များတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါနိုင်ကြောင်း သတိပြုပါ။ မူလစာတမ်းကို မူလဘာသာစကားဖြင့် တင်ထားသည့် အခြားစာရွက်ကို တရားဝင် အရင်းအမြစ်အဖြစ် သတ်မှတ်သင့်ပါသည်။ အရေးကြီးသော သတင်းအချက်အလက်များအတွက် လူ့ပညာရှင်များမှ ပြုလုပ်သော ပရော်ဖက်ရှင်နယ် ဘာသာပြန်ချက်ကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုရာမှ ဖြစ်ပေါ်လာသော နားမလည်မှုများ သို့မဟုတ္ အဓိပ္ပာယ်မှားဖွားခြင်းများအတွက် ကျွန်တော်/ကျွန်မတို့သည် တာဝန်မယူပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
